# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Course:** LMA — Assignment 1

**Goal:** build two retrieval pipelines over *the same* chunked corpus and contrast them:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-Ranked) |
|---|---|---|
| Retrieve | Bi-encoder, Top 10 | Bi-encoder, Top 10 (identical) |
| Select | Top 3 as-is | Cross-encoder re-sorts, Top 3 |
| Generate | LLM -> **Answer 1** | Same prompt -> **Answer 2** |

**Document:** _TODO: name + source URL of the PDF you chose_

---
## 1. Setup & Environment

Local-only stack: bi-encoder and cross-encoder from `sentence-transformers`, generation via Ollama.

In [ ]:
import json
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import ollama

In [ ]:
# ---- Configuration: single source of truth for the whole notebook ----

PDF_PATH = Path("data/<your-document>.pdf")

# Bi-encoder: independent embeddings, used to build the vector index
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Cross-encoder: joint (query, chunk) scoring, used only to re-rank
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Generator (local, via Ollama)
LLM_MODEL = "qwen2.5:7b-instruct"

# Chunking
CHUNK_SIZE = 800      # characters (or tokens, if you chunk by token)
CHUNK_OVERLAP = 150

# Retrieval depths mandated by the assignment
TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_N_CONTEXT = 3     # chunks actually shown to the LLM

RANDOM_SEED = 42

In [ ]:
def check_environment() -> None:
    """Sanity-check that Ollama is up, LLM_MODEL is pulled, and MPS/CPU device is known.

    TODO:
      - ollama.list() -> confirm LLM_MODEL is present
      - print torch device that sentence-transformers will use (mps on Apple Silicon)
      - print library versions so the report can state the exact setup
    """
    raise NotImplementedError


check_environment()

---
## 2. Load & Chunk the Document

Both pipelines must share **the exact same corpus**, so chunking happens once, here.

Design notes to decide (and justify in the report):
- Chunk boundaries: fixed-size with overlap vs. structure-aware (split on section / rule numbers).
- Keep per-chunk metadata (page number, section heading) — you need it to describe rank shifts.

In [ ]:
def load_pdf_text(pdf_path: Path) -> list[dict]:
    """Extract text from the PDF, one record per page.

    Returns: [{"page": int, "text": str}, ...]

    TODO:
      - PdfReader(pdf_path), iterate .pages, .extract_text()
      - light cleanup: collapse whitespace, drop repeated headers/footers
    """
    raise NotImplementedError

In [ ]:
@dataclass
class Chunk:
    """One indexed unit of the corpus."""
    id: str
    text: str
    page: int
    section: str | None = None


def chunk_pages(pages: list[dict], chunk_size: int, overlap: int) -> list[Chunk]:
    """Split page text into overlapping chunks, preserving page/section metadata.

    TODO:
      - sliding window of chunk_size with `overlap` carry-over
      - prefer splitting on paragraph/sentence boundaries over mid-word cuts
      - assign stable ids (e.g. "c0042") -- you will cite these in the rank-shift table
    """
    raise NotImplementedError

In [ ]:
# Build the corpus and eyeball it before indexing.
# TODO: pages = load_pdf_text(PDF_PATH); chunks = chunk_pages(...)
#       then check: len(chunks), length distribution, print 2-3 samples

---
## 3. Index the Corpus (Bi-Encoder)

The bi-encoder embeds each chunk **independently** and offline. The query is embedded the same way at
search time, and relevance is a similarity between two fixed vectors — the model never sees the query
and the chunk together.

In [ ]:
def embed_texts(model: SentenceTransformer, texts: list[str]) -> np.ndarray:
    """Encode texts into a (n, dim) float32 matrix.

    TODO:
      - model.encode(..., batch_size=..., normalize_embeddings=True, show_progress_bar=True)
      - normalising means inner product == cosine similarity
    """
    raise NotImplementedError


def build_vector_store(chunks: list[Chunk], embeddings: np.ndarray):
    """Create a Chroma collection holding the chunk vectors, text, and metadata.

    TODO:
      - chromadb.Client() (or PersistentClient to avoid re-embedding on restart)
      - create_collection(..., metadata={"hnsw:space": "cosine"})
      - collection.add(ids=..., embeddings=..., documents=..., metadatas=...)
    """
    raise NotImplementedError

In [ ]:
# TODO: load the bi-encoder, embed all chunks (time this -- it is your one-off indexing cost),
#       build the collection, and assert collection.count() == len(chunks)

---
## 4. Pipeline 1 — Baseline Retrieval

Retrieve Top 10 by vector similarity, hand the **Top 3** straight to the LLM -> **Answer 1**.

In [ ]:
@dataclass
class Candidate:
    """A retrieved chunk with whatever scores it has accumulated."""
    chunk_id: str
    text: str
    page: int
    dense_rank: int            # 1-based rank from the bi-encoder
    dense_score: float
    cross_score: float | None = None
    cross_rank: int | None = None


def retrieve_baseline(query: str, collection, embed_model, k: int = TOP_K_RETRIEVE) -> list[Candidate]:
    """Embed the query, return the Top-k candidates ordered by vector similarity.

    TODO:
      - embed query with the SAME model/normalisation used for the chunks
      - collection.query(query_embeddings=..., n_results=k)
      - map results into Candidate objects, filling dense_rank / dense_score
      - note: Chroma returns cosine *distance*; convert if you want similarity
    """
    raise NotImplementedError

In [ ]:
PROMPT_TEMPLATE = """\
# TODO: write ONE prompt template and reuse it byte-for-byte in both pipelines --
# otherwise the comparison is confounded.
#
# It should:
#   - instruct the model to answer ONLY from the provided context
#   - require it to say so explicitly when the context is insufficient (so you can
#     catch hallucinations in the failure-mode analysis)
#   - require citations by chunk id / page
#   - leave slots for {context} and {question}
"""


def format_context(candidates: list[Candidate]) -> str:
    """Render the selected chunks into the {context} slot, labelled with id and page."""
    raise NotImplementedError


def generate_answer(question: str, candidates: list[Candidate], model: str = LLM_MODEL) -> str:
    """Call the local LLM with the shared prompt template.

    TODO:
      - ollama.chat(model=model, messages=[...], options={"temperature": 0, "seed": RANDOM_SEED})
      - temperature 0 so Answer 1 vs Answer 2 differences come from the CONTEXT, not sampling
    """
    raise NotImplementedError

In [ ]:
def run_pipeline_1(query: str) -> dict:
    """Baseline: Top-10 dense retrieval -> Top-3 as-is -> Answer 1.

    Returns a dict with: candidates (all 10), context_used (3), answer, and timings
    (retrieval_s, generation_s) -- you need these for the latency table in section 7.
    """
    raise NotImplementedError

---
## 5. Pipeline 2 — Cross-Encoder Re-Ranking

Same Top-10 candidate set. Each `(query, chunk)` pair goes through the cross-encoder **jointly**, so
self-attention runs across query and chunk tokens together. Re-sort by that score, take the new Top 3,
and use the **identical prompt** -> **Answer 2**.

In [ ]:
def rerank(query: str, candidates: list[Candidate], cross_encoder: CrossEncoder) -> list[Candidate]:
    """Score every (query, chunk) pair jointly and re-sort descending.

    TODO:
      - pairs = [(query, c.text) for c in candidates]   # all 10, not just the top few
      - cross_encoder.predict(pairs)
      - write cross_score back onto each Candidate, sort desc, fill cross_rank
      - keep dense_rank intact -- the rank SHIFT is the headline result of this assignment
    """
    raise NotImplementedError


def run_pipeline_2(query: str) -> dict:
    """Re-ranked: Top-10 dense -> cross-encoder re-sort -> Top-3 -> Answer 2.

    Same return shape as run_pipeline_1, plus a rerank_s timing.
    """
    raise NotImplementedError

In [ ]:
# Smoke test on one straightforward query: both pipelines should answer it correctly.
# This establishes that re-ranking is not *breaking* the easy cases.
# TODO: pick a query whose answer you can verify by hand in the PDF, run both, print side by side.

---
## 6. Failure Mode Analysis (Stress Testing)

Find **at least two** queries where the baseline stumbles and re-ranking recovers. Target these
patterns (the assignment names them explicitly):

| Pattern | What you are engineering |
|---|---|
| **Keyword trap / distractor** | Query shares vocabulary with an irrelevant passage; the real answer uses synonyms |
| **Exception / corner case** | Baseline finds the general rule, misses the "except when…" clause elsewhere |
| **Negation & constraints** | "not allowed", "prohibited", role limits — where embeddings blur positive vs. negative |
| **Buried gold** | The vital chunk sits at dense rank 4–10 and the cross-encoder promotes it into the Top 3 |

**Method:** read your document first and find the clauses that *conflict* or *qualify* each other.
Write queries against those. Queries engineered from the document beat queries invented in the abstract.

In [ ]:
FAILURE_QUERIES = [
    # {
    #     "id": "F1",
    #     "pattern": "keyword_trap",
    #     "query": "...",
    #     "gold_chunk_ids": ["c0137"],   # fill in after you verify by hand
    #     "gold_fact": "the clause the answer MUST contain",
    #     "why_baseline_fails": "your hypothesis, written BEFORE you run it",
    # },
]

In [ ]:
def analyse_query(case: dict) -> dict:
    """Run both pipelines on one failure case and collect everything the report needs.

    TODO return:
      - the two answers
      - the rank shift for each gold chunk: dense_rank -> cross_rank  (e.g. #8 -> #1)
      - whether each gold chunk made the Top 3 under each pipeline
      - timings
    """
    raise NotImplementedError


def rank_shift_table(results: list[dict]) -> pd.DataFrame:
    """Tabulate: query | chunk id | dense rank | cross rank | in Top-3 before? | after?

    This table is the core evidence for report section 2.
    """
    raise NotImplementedError

In [ ]:
# TODO: run every case in FAILURE_QUERIES, display the rank-shift table, and print
#       Answer 1 vs Answer 2 side by side for each.
#
# When comparing answers, label the failure precisely -- the report asks which of these happened:
#   hallucination  |  omitted essential clause  |  misstated the rule

---
## 7. Latency Profiling

Separate the stages, or the numbers mean nothing. Report **retrieval** and **re-ranking** apart from
**generation** — LLM time dwarfs both and would hide the effect you are measuring.

In [ ]:
def profile_pipelines(queries: list[str], n_runs: int = 5) -> pd.DataFrame:
    """Time each stage across several queries.

    Columns to produce: stage | mean_ms | p50 | p95
    Stages: embed_query | vector_search | cross_encode_10 | llm_generate

    TODO:
      - discard a warm-up run (first call pays model load + lazy init)
      - time.perf_counter() around each stage
      - also record the one-off indexing cost from section 3 for context
    """
    raise NotImplementedError

In [ ]:
# Optional but persuasive for the scaling question in section 8:
# measure cross-encoder latency vs. number of pairs (10, 50, 100, 500, len(chunks))
# and plot it. The curve IS the answer to "why not cross-encode everything?"

---
## 8. Report Notes

The report is submitted alongside this notebook and **must be in your own words** — the assignment
says explicitly that it should not be AI-generated. Use this section only to park the raw numbers and
observations you will write up.

**1. Architecture comparison.** Bi-encoder vs. cross-encoder: where the query and document meet in
each, and what that costs. Explain *why* independent embeddings can blur negation.

**2. Qualitative & rank-shift analysis.** Your table from section 6, plus for each failure query: what
Answer 1 got wrong, and which specific chunk promotion fixed it.

**3. Latency profiling.** Your table from section 7.

**4. Scaling question.** Why not cross-encode the whole collection from the start? Ground it in your
own measured numbers: chunks in your corpus x per-pair cross-encoder cost, and note that chunk
embeddings precompute once while cross-encoder scores cannot be cached per query.

**Checklist before submitting**
- [ ] Single well-commented `.ipynb`, run top to bottom with outputs saved
- [ ] Public/shared GitHub repo, PDF (or its source link) included
- [ ] >= 2 failure queries with documented rank shifts
- [ ] Identical prompt + identical candidate set across both pipelines
- [ ] Latency table
- [ ] Report written by you